Get the aligned numbers and merge with the data:

In [1]:
import pandas as pd
import re
import os
import paramiko
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from itertools import combinations
from pprint import pprint
from itertools import chain
from sqlalchemy import create_engine
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy()) 

data = pd.read_csv(r"c:\Users\glj523\OneDrive - University of Copenhagen\all_qc_data_ill_comp.csv")
data = data.drop(columns=["Unnamed: 0"])
data.insert(loc=1, column="Lane", value=data["qc_type"].apply(lambda x: x.split("_")[0]))
data.insert(loc=2, column="Read Type", value=data["qc_type"].apply(lambda x: x.split("_")[1] if len(x.split("_")) > 1 else np.NaN))

reads_aligned_nsx = pd.DataFrame(columns=['1', '2', 'ttag', '3', 'eDNA Concentration', '4', 'library_id', '6', 'reads_aligned'])
reads_aligned_ns6 = pd.DataFrame(columns=['flowcell', '2', 'library_id', '3', 'reads_aligned'])

hostname = 'dandycomp04fl'  # Replace with your server's IP or hostname
port = 22                       # Usually 22 for SSH
username = 'glj523'      # Replace with your username
password = 'Wtcantfw36c!123'      # Replace with your password

try:
    ssh.connect(hostname, port=port, username=username, password=password)
    _, out, _ = ssh.exec_command(f"ls ~/bash_scripts/get_total_aligned_reads/from_non_production/results")
    files = out.read().decode().splitlines()
    for i, file in enumerate(files):
        _, out, _ = ssh.exec_command(f"cat ~/bash_scripts/get_total_aligned_reads/from_non_production/results/{file}")
        res = out.read().decode()
        new_row = str(file).split('_')
        new_row.append(int(res))
        reads_aligned_nsx.loc[i] = new_row
    
    _, out, _ = ssh.exec_command(f"ls ~/bash_scripts/get_total_aligned_reads/from_production/results")
    files = out.read().decode().splitlines()
    for i, file in enumerate(files):
        _, out, _ = ssh.exec_command(f"cat ~/bash_scripts/get_total_aligned_reads/from_production/results/{file}")
        res = out.read().decode()
        new_row = str(file).split('_')
        new_row.append(int(res) / 4)  # Divide by four because otherwise its not comparible with nsx that has 1 value per lane
        reads_aligned_ns6.loc[i] = new_row
finally:
    ssh.close
    
reads_aligned_ns6['Protocol'] = reads_aligned_ns6['flowcell']
reads_aligned_ns6['Protocol'] = reads_aligned_ns6['Protocol'].replace('AHNLW5DSX5', 'Double').replace('BH5F5KDSX7', 'Single')
reads_aligned_ns6['Platform'] = 'NovaSeq6'
reads_aligned_ns6['eDNA Concentration'] = 'Unknown?'
reads_aligned_nsx['Protocol'] = reads_aligned_nsx['ttag']
reads_aligned_nsx['Protocol'] = reads_aligned_nsx['Protocol'].replace('WBDQ4', 'Single').replace('AOZCK', 'Double')
reads_aligned_nsx['Platform'] = 'NovaSeqX'

controls = ['LV7009026112',
            'LV7009026448',
            'LV7009026461',
            'LV7009026511',
            'LV7009026519',
            'LV7008960913',
            'LV7009026337',
            'LV7009026399',
            'LV7009026461',
            'LV7009026519',
            'LV7008960913']

assert len(reads_aligned_nsx[~reads_aligned_nsx['library_id'].isin(controls)]) / 4 == len(data.library_id.unique())

reads_aligned_nsx = reads_aligned_nsx[~reads_aligned_nsx['library_id'].isin(controls)]

assert len(reads_aligned_ns6[~reads_aligned_ns6['library_id'].isin(controls)]) == len(data.library_id.unique())
reads_aligned_ns6 = reads_aligned_ns6[~reads_aligned_ns6['library_id'].isin(controls)]
reads_aligned_nsx = reads_aligned_nsx[['library_id', 'reads_aligned', 'Protocol', 'Platform', 'eDNA Concentration']]
reads_aligned_ns6 = reads_aligned_ns6[['library_id', 'reads_aligned', 'Protocol', 'Platform', 'eDNA Concentration']]
concd_reads_aligned = pd.concat([reads_aligned_ns6, reads_aligned_nsx])
assert len(concd_reads_aligned) == (115*4) + 115
merge_test = data.merge(concd_reads_aligned, on=['library_id', 'Protocol', 'Platform', 'eDNA Concentration'])
assert ((merge_test
  .astype(str)
 .drop('reads_aligned', axis=1)[data.columns]
 .sort_values(by=list(data.columns))
 .reset_index(drop=True)) 
 ==
(data[data.columns].astype(str)
 .sort_values(by=list(data.columns))
 .reset_index(drop=True))).all().all()

for lib_id in data.library_id.unique():
    test1 = concd_reads_aligned[concd_reads_aligned['library_id'] == lib_id].sort_values('eDNA Concentration').reset_index(drop=True)
    test2 = merge_test[(merge_test['library_id'] == lib_id) & (merge_test['Read Type'] == 'collapsed')][['library_id', 'reads_aligned', 'Protocol', 'Platform', 'eDNA Concentration']].drop_duplicates().sort_values('eDNA Concentration').reset_index(drop=True)
    assert (test1 == test2).all().all()
  
data = data.merge(concd_reads_aligned, on=['library_id', 'Protocol', 'Platform', 'eDNA Concentration'])
data.to_csv(r'C:\Users\glj523\OneDrive - University of Copenhagen\data\illumina_analysis\data.csv', index=False)

C:\Users\glj523\AppData\Local\Temp\ipykernel_15080\2374321924.py:15: DtypeWarning: Columns (35) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(r"c:\Users\glj523\OneDrive - University of Copenhagen\all_qc_data_ill_comp.csv")


ValueError: cannot set a row with mismatched columns